In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import torch.nn.functional as F

from transformers import (
    AutoProcessor,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments
)

from knowledge_distillation import VisionTextDataset

In [ ]:
teacher_name = "meta-llama/Llama-3.2-11B-Vision"
student_name = "meta-llama/Llama-3.2-3B-Vision"

processor = AutoProcessor.from_pretrained(teacher_name)

teacher = AutoModelForCausalLM.from_pretrained(
    teacher_name,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_8bit=True,  # or 4bit (bitsandbytes)
)
teacher.eval()  # IMPORTANT

student = AutoModelForCausalLM.from_pretrained(
    student_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./distilled-llama-vision",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    fp16=True,
    logging_steps=10,
    save_steps=500,
    learning_rate=2e-5,
    report_to="none"
)

trainer = DistillationTrainer(
    model=student,
    teacher=teacher,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=processor.tokenizer
)

trainer.train()